<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/Kimi_K2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading packages

In [ ]:
import torch
import torch.nn as nn
import math
from transformers import AutoTokenizer
import torch.nn.functional as F

# Model args

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf", token='TOKEN')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

1

In [ ]:
class ModelArgs:
    block_size = 256
    batch_size = 32
    embeddings_dims = 512
    attn_dropout = 0.1
    no_of_heads = 8 #IMP needs to be thoroughly calculated
    dropout = 0.1
    epochs = 1
    max_lr = 6e-4
    no_of_decoder_layers = 8 #IMP needs to be thoroughly calculated
    weight_decay_optim = 0.1
    beta_1 = 0.9
    beta_2 = 0.95
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    vocab_size =len(tokenizer) ## Make it available during training
    base_freq=10000
    # s = 1.0
    experts=16
    shared_experts=1
    clip = 1.0
    top_experts=4
    noisy_topk = False
    use_checkpointing = False
    use_liger = False  # Use Liger kernels for optimized operations
    use_shared_expert = True  # Enable/disable shared expert
    ignore_pad_token_in_loss = True  # Whether to ignore padding tokens in loss calculation
    eps: float = 1e-8
    loss_scale = 0.3
    useauxFreeLoadBalancingLoss = True
    aux_free_bias_update_rate = 0.001
    mtp_heads = 1  # Multi-token prediction heads
    latent_dim = 64  # Latent dimension for attention
    rope_dim=32

# RMS Norm

In [ ]:
class RMSNorm(nn.Module):
  def __init__(self, dim, eps=1e-8, device=None):
    super().__init__()
    self.gamma=nn.Parameter(torch.ones(dim, device=device))
    self.dim=dim
    self.eps=eps

  def norm(self, x):
    val=x.pow(2).mean(-1, keepdim=True)
    val=torch.sqrt(val+self.eps)
    return x/val
  def forward(self, x):
    return self.gamma*self.norm(x)



# KVCache

In [ ]:
class KVCache(nn.Module):
  def __init__(self, num_layers=ModelArgs.no_of_decoder_layers):
    super().__init__()
    self.num_layers=num_layers
    self.cache=[None]*self.num_layers
  def num_items(self):
    return sum(1 for item in self.cache if item is not None)
  def add_item(self, latent_matrix, layer_idx):
    if layer_idx>=self.num_layers:
      raise IndexError(f"layer_idx {layer_idx} is out of range for KV cache with {len(self.latent_cache)} layers.")

    if self.cache[layer_idx] is None:
      self.cache[layer_idx]=latent_matrix
    else:
      self.cache[layer_idx]=torch.cat((self.cache[layer_idx], latent_matrix), dim=1)
  def get_item(self, layer_idx):
    if layer_idx>=self.num_layers:
      raise IndexError(f"layer_idx {layer_idx} is out of range for KV cache with {len(self.latent_cache)} layers.")
    return self.cache[layer_idx]
  def cleanup(self):
    self.cache=[None]*self.num_layers

### KVCache modified

In [ ]:
class KVCache(nn.Module):
    def __init__(self, num_layers=ModelArgs.no_of_decoder_layers):
        super().__init__()
        self.num_layers = num_layers
        self.latent_cache = [None] * self.num_layers
        self.pe_cache = [None] * self.num_layers  # Add PE cache

    def num_items(self):
        return sum(1 for item in self.latent_cache if item is not None)

    def add_item(self, latent_matrix, pe_rotated, layer_idx):
        """
        latent_matrix: (B, S, latent_dim)
        pe_rotated: (B, S, rope_dim) - already rotated
        """
        if layer_idx >= self.num_layers:
            raise IndexError(f"layer_idx {layer_idx} out of range")

        # Cache latent
        if self.latent_cache[layer_idx] is None:
            self.latent_cache[layer_idx] = latent_matrix
        else:
            self.latent_cache[layer_idx] = torch.cat([self.latent_cache[layer_idx], latent_matrix], dim=1)

        # Cache rotated PE
        if self.pe_cache[layer_idx] is None:
            self.pe_cache[layer_idx] = pe_rotated
        else:
            self.pe_cache[layer_idx] = torch.cat([self.pe_cache[layer_idx], pe_rotated], dim=1)

    def get_item(self, layer_idx):
        """Returns (latent, pe_rotated) tuple or (None, None)"""
        if layer_idx >= self.num_layers:
            raise IndexError(f"layer_idx {layer_idx} out of range")
        return self.latent_cache[layer_idx], self.pe_cache[layer_idx]

    def cleanup(self):
        self.latent_cache = [None] * self.num_layers
        self.pe_cache = [None] * self.num_layers

# ROPE

In [ ]:
class RoPE(nn.Module):
  def __init__(self, dims=ModelArgs.embeddings_dims, base_freq=ModelArgs.base_freq, max_seq_len=ModelArgs.block_size):
    super().__init__()
    self.dims=dims
    self.base_freqs=base_freq
    self.batch_size=ModelArgs.batch_size
    self.max_seq_len=max_seq_len

    assert self.dims%2==0, "Head dim must be div by 2"

    theta_num=torch.arange(0, self.dims, 2).float()
    theta=1.0/(self.base_freqs**(theta_num/self.dims))
    self.register_buffer("theta", theta)
    positions=torch.arange(0, self.max_seq_len, dtype=torch.float)
    angles=positions.unsqueeze(1)*theta.unsqueeze(0)
    self.register_buffer("cosine_cached", torch.cos(angles))
    self.register_buffer("sine_cached", torch.sin(angles))
  def forward(self, x, start_pos=0):
    b,s,h,d=x.shape
    x=x.view(b,s,h,d//2, 2)
    cos=self.cosine_cached[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)
    sin=self.sine_cached[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)

    x_rot=torch.stack([
        x[..., 0]*cos-x[..., 1]*sin,
        x[..., 0]*sin+x[..., 1]*cos
    ], dim=-1)
    return x_rot.view(b,s,h,d)


# Swish

In [ ]:
class Swish(nn.Module):
  def __init__(self):
    super().__init__()
    self.sigmoid=nn.Sigmoid()
  def forward(self, x):
    return x*self.sigmoid(x)

# Deepseek MOE

## Experts

In [ ]:
class Expert(nn.Module):
  def __init__(self, block_size=ModelArgs.block_size, embeddings_dims=ModelArgs.embeddings_dims, device=ModelArgs.device):
    super().__init__()
    self.hidden_dims=((embeddings_dims*2)*4)//3
    self.block_size=block_size
    self.embeddings_dims=embeddings_dims
    self.device=device
    self.linear1=nn.Linear(self.embeddings_dims, self.hidden_dims, bias=False, device=self.device)
    self.linear2=nn.Linear(self.embeddings_dims, self.hidden_dims,bias=False, device=self.device)
    self.linear3=nn.Linear(self.hidden_dims, self.embeddings_dims, bias=False, device=self.device)
    self.swish=Swish()

  def forward(self, x):
    x1=self.linear1(x)
    x2=self.linear2(x)
    hidden=torch.mul(self.swish(x1), x2)
    return self.linear3(hidden)

## Main logic

In [ ]:
class DeepSeekMOE(nn.Module):
  def __init__(self, block_size=ModelArgs.block_size, embeddings_dims=ModelArgs.embeddings_dims, device=ModelArgs.device):
    super().__init__()
    self.block_size=block_size
    self.embeddings_dims=embeddings_dims
    self.device=device
    self.aux_free_bias_update_rate=ModelArgs.aux_free_bias_update_rate
    self.gate=nn.Linear(self.embeddings_dims, ModelArgs.experts, bias=False, device=self.device)
    self.experts=nn.ModuleList([Expert(self.block_size, self.embeddings_dims, self.device) for _ in range(ModelArgs.experts)])
    if ModelArgs.use_shared_expert:
      self.shared_expert=Expert(self.block_size, self.embeddings_dims, self.device)
    if ModelArgs.noisy_topk==True and ModelArgs.use_checkpointing==False:
      self.noisy=nn.Linear(self.embeddings_dims, ModelArgs.experts, bias=False, device=self.device)
    else:
      self.noisy=None
    if ModelArgs.useauxFreeLoadBalancingLoss:
      self.register_buffer("routing_bias", torch.zeros(ModelArgs.experts, device=self.device))

  def forward(self,x):
    batch_size, seq_len, embed_dims=x.shape
    self.gate_op=self.gate(x)
    if ModelArgs.noisy_topk==True:
      self.noisy_op=self.noisy(x)
      gaussian_noise=torch.normal(0,1,size=self.gate_op.shape, device=self.device)
      self.noisy_router=torch.nn.functional.softplus(self.gate_op+gaussian_noise)
      self.gate_op+=self.noisy_router


    shared_op=0
    output=0
    if ModelArgs.useauxFreeLoadBalancingLoss:
      self.gate_op+=self.routing_bias

    top_k=ModelArgs.top_experts
    top_k_vals, top_k_indices=torch.topk(self.gate_op, k=top_k, dim=-1)
    masked=torch.full_like(self.gate_op, fill_value=float("-inf"), device=self.device)
    masked_values=masked.scatter_(dim=-1, index=top_k_indices, src=top_k_vals)
    probs=torch.nn.functional.softmax(masked_values, dim=-1)

    output=torch.zeros_like(x)
    flat_x=x.view(-1, x.size(-1))

    ## Get through all experts
    for i in range(ModelArgs.experts):
      experts_i_is_chosen_mask=(top_k_indices==i).any(dim=-1)

      if not experts_i_is_chosen_mask.any():
        continue
      flat_expert_i_is_chosen_mask=experts_i_is_chosen_mask.view(-1)
      selected_expert_input=flat_x[flat_expert_i_is_chosen_mask]
      expert_output_for_selected=self.experts[i](selected_expert_input)
      probs_for_exp_i=probs[:,:,i]
      token_weights=probs_for_exp_i[experts_i_is_chosen_mask]
      token_weights=token_weights.unsqueeze(-1)
      weighted_expert_op=expert_output_for_selected*token_weights

      temp_contri_from_x=torch.zeros_like(x)
      temp_contri_from_x.masked_scatter_(
          experts_i_is_chosen_mask.unsqueeze(-1).expand_as(x),
          weighted_expert_op
      )
      output+=temp_contri_from_x

      output+=shared_op

      if ModelArgs.useauxFreeLoadBalancingLoss and self.training:
        with torch.no_grad():
          ci=probs.sum(dim=(0,1))
          ci_mean=ci.mean()

          error=ci_mean-ci
          self.update=self.aux_free_bias_update_rate*torch.sign(error)
          self.routing_bias.add(self.update)
    return output


# MultiHead Latent Attention

In [ ]:
class MultiHeadLatentAttention(nn.Module):
  def __init__(self, layer_idx, block_size=ModelArgs.block_size, embeddings_dims=ModelArgs.embeddings_dims, device=ModelArgs.device):
    super().__init__()
    self.block_size=block_size
    self.embeddings_dims=embeddings_dims
    self.device=device
    self.latent_dims=ModelArgs.latent_dim
    self.no_of_heads=ModelArgs.no_of_heads
    self.attn_dropout=ModelArgs.attn_dropout
    assert self.embeddings_dims%self.no_of_heads==0, "Head dim must be div by no of heads"
    self.head_dim=self.embeddings_dims//self.no_of_heads
    self.dropout=nn.Dropout(self.attn_dropout)
    self.Wq=nn.Linear(self.embeddings_dims, self.head_dim*self.no_of_heads, device=self.device, bias=False)
    self.Wk=nn.Linear(self.latent_dims, self.head_dim*self.no_of_heads, device=self.device, bias=False)
    self.Wv=nn.Linear(self.latent_dims, self.head_dim*self.no_of_heads, device=self.device, bias=False)

    self.Wkv=nn.Linear(self.embeddings_dims,self.latent_dims, device=self.device, bias=False)

    self.rope=RoPE(dims=self.head_dim, base_freq=ModelArgs.base_freq, max_seq_len=self.block_size)
    self.layer_idx=layer_idx

  def forward(self,x, kv_cache=None, mask=None):
    batch_size, seq_len, embed_dims=x.shape
    q=self.Wq(x)
    self.latent_mat=self.Wkv(x)
    if kv_cache is not None:
      kv_cache.add_item(self.latent_mat, self.layer_idx)
      self.latent_mat=kv_cache.get_item(self.layer_idx)
    k=self.Wk(self.latent_mat)
    v=self.Wv(self.latent_mat)
    kv_len=self.latent_mat.shape[1]

    q=q.view(batch_size, seq_len, self.no_of_heads, self.head_dim)
    k=k.view(batch_size, kv_len, self.no_of_heads, self.head_dim)
    v=v.view(batch_size, kv_len, self.no_of_heads, self.head_dim)

    start_pos=max(0, kv_len-seq_len)
    q=self.rope(q, start_pos)
    k=self.rope(k)

    q=q.transpose(1,2)
    k=k.transpose(1,2)
    v=v.transpose(1,2)

    attn_weights=torch.matmul(q, k.transpose(-2,-1))/self.head_dim**0.5
    if mask is not None:
      attn_weights=attn_weights.masked_fill(mask==0, float("-inf"))

    causal_mask=torch.ones(seq_len, kv_len, device=self.device)
    if seq_len==kv_len:
      causal_mask=torch.tril(causal_mask)
    else:
      pass
    causal_mask.unsqueeze(0)
    causal_mask.unsqueeze(1)
    attn_weights=attn_weights.masked_fill(causal_mask==0, float("-inf"))

    attn_weights=torch.nn.functional.softmax(attn_weights, dim=-1)
    attn_weights=self.dropout(attn_weights)
    attn_output=torch.matmul(attn_weights, v)
    attn_output=attn_output.transpose(1,2)
    attn_output=attn_output.contiguous().view(batch_size, seq_len, embed_dims)
    return attn_output, kv_cache




### MLA modified

In [ ]:
class MultiHeadLatentAttention(nn.Module):
  def __init__(self, layer_idx, block_size=ModelArgs.block_size, embeddings_dims=ModelArgs.embeddings_dims, device=ModelArgs.device):
    super().__init__()
    self.block_size=block_size
    self.embeddings_dims=embeddings_dims
    self.device=device
    self.latent_dims=ModelArgs.latent_dim
    self.no_of_heads=ModelArgs.no_of_heads
    self.attn_dropout=ModelArgs.attn_dropout
    self.rope_dim=ModelArgs.rope_dim
    assert self.embeddings_dims%self.no_of_heads==0, "Head dim must be div by no of heads"
    self.head_dim=self.embeddings_dims//self.no_of_heads

    self.nope_dim=self.head_dim-self.rope_dim #32



    self.dropout=nn.Dropout(self.attn_dropout)
    self.Wq=nn.Linear(self.embeddings_dims, self.head_dim*self.no_of_heads, device=self.device, bias=False)
    self.Wk_nope=nn.Linear(self.latent_dims, self.nope_dim*self.no_of_heads, device=self.device, bias=False)
    self.Wv=nn.Linear(self.latent_dims, self.head_dim*self.no_of_heads, device=self.device, bias=False)

    self.Wkv=nn.Linear(self.embeddings_dims,self.latent_dims + self.rope_dim, device=self.device, bias=False)

    self.rope=RoPE(dims=self.rope_dim, base_freq=ModelArgs.base_freq, max_seq_len=self.block_size)
    self.layer_idx=layer_idx

  def forward(self,x, kv_cache=None, mask=None):
    batch_size, seq_len, embed_dims=x.shape
    q=self.Wq(x).view(batch_size, seq_len, self.no_of_heads, self.head_dim)
    q_nope, q_rope=q.split([self.nope_dim, self.rope_dim], dim=-1)

    kv_op=self.Wkv(x)
    curr_latent, k_rope_curr=kv_op.split([self.latent_dims, self.rope_dim], dim=-1)

    if kv_cache is not None:
      latent_mat, pe_rotated=kv_cache.get_item(self.layer_idx)
      if latent_mat is not None:
        start_pos=latent_mat.shape[1]
      else:
        start_pos=0
    else:
      start_pos=0

    k_rope_curr=k_rope_curr.unsqueeze(2)
    k_rope_rotated=self.rope(k_rope_curr, start_pos)
    k_rope_rotated=k_rope_rotated.squeeze(2)

    q_rope=self.rope(q_rope, start_pos)

    if kv_cache is not None:
      kv_cache.add_item(curr_latent, k_rope_rotated, self.layer_idx)
      latent_mat, pe_mat=kv_cache.get_item(self.layer_idx)
    else:
      latent_mat=curr_latent
      pe_mat=k_rope_rotated

    kv_len=latent_mat.shape[1]
    k_nope=self.Wk_nope(latent_mat)
    k_nope=k_nope.view(batch_size, kv_len, self.no_of_heads, self.nope_dim)

    v=self.Wv(latent_mat)
    v=v.view(batch_size, kv_len, self.no_of_heads, self.head_dim)

    q_nope=q_nope.transpose(1,2)
    q_rope=q_rope.transpose(1,2)
    k_nope=k_nope.transpose(1,2)
    v=v.transpose(1,2)


    ## Attention

    scores_nope=torch.matmul(q_nope, k_nope.transpose(-2,-1))
    scores_rope=torch.einsum("bhsd, btd->bhst", q_rope, pe_mat)

    attn_weights=(scores_nope+scores_rope)/(self.head_dim**0.5)

    if mask is not None:
      attn_weights=attn_weights.masked_fill(mask==0, float("-inf"))

    causal_mask=torch.ones(seq_len, kv_len, device=self.device)
    if seq_len==kv_len:
      causal_mask=torch.tril(causal_mask)
    causal_mask=causal_mask.unsqueeze(0)
    causal_mask=causal_mask.unsqueeze(1)
    attn_weights=attn_weights.masked_fill(causal_mask==0, float("-inf"))
    attn_weights=torch.nn.functional.softmax(attn_weights, dim=-1)
    attn_weights=self.dropout(attn_weights)
    attn_output=torch.matmul(attn_weights, v)
    attn_output=attn_output.transpose(1,2)
    attn_output=attn_output.contiguous().view(batch_size, seq_len, embed_dims)
    return attn_output, kv_cache

# Decoder Layer

In [ ]:
class Decoder_Layer(nn.Module):
  def __init__(self, layer_idx=None, embed_dims=ModelArgs.embeddings_dims, device=ModelArgs.device, p_dropout=ModelArgs.dropout):
    super().__init__()
    self.layer_idx=layer_idx
    self.embed_dims=embed_dims
    self.device=device
    self.MLA=MultiHeadLatentAttention(self.layer_idx, embeddings_dims=self.embed_dims, device=self.device)
    self.norm1=RMSNorm(self.embed_dims, eps=ModelArgs.eps, device=device)
    self.norm2=RMSNorm(self.embed_dims, eps=ModelArgs.eps, device=device)
    self.MOE=DeepSeekMOE(embeddings_dims=self.embed_dims, device=self.device)
    self.layer_idx=layer_idx
    self.dropout=nn.Dropout(p=p_dropout)

  def forward(self, x, mask=None, kv_cache=None):
    out,kv_cache=self.MLA(self.norm1(x), mask=mask, kv_cache=kv_cache)
   # print(out)
    x=x+self.dropout(out)
    x=self.norm2(x)
    out=self.MOE(x)
    x=x+self.dropout(out)
    return x, kv_cache

# Block

In [ ]:
class Block(nn.Module):
  def __init__(self, device=ModelArgs.device, embeddings_dims=ModelArgs.embeddings_dims, no_of_decoder_layers=ModelArgs.no_of_decoder_layers, block_size=ModelArgs.block_size, vocab_size=ModelArgs.vocab_size, dropout=ModelArgs.dropout):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size, embeddings_dims, dtype=torch.float32, device=device)
    self.device=device
    self.embeddings_dims=embeddings_dims
    self.no_of_decoder_layers=no_of_decoder_layers
    self.block_size=block_size
    self.vocab_size=vocab_size
    self.dropout=nn.Dropout(dropout)
    self.norm=RMSNorm(embeddings_dims, device=self.device)
    self.layers=nn.ModuleList([Decoder_Layer(layer_idx=i, embed_dims=self.embeddings_dims, device=self.device) for i in range(self.no_of_decoder_layers)])
    self.linear_layer=nn.Linear(self.embeddings_dims, self.vocab_size, bias=False, dtype=torch.float32, device=self.device)
    self.embedding.weight=self.linear_layer.weight
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


  def forward(self, x, mask=None,kv_cache=None):
    for layer in self.layers:
      x, kv_cache=layer(x, mask=mask, kv_cache=kv_cache)
    x=self.dropout(x)
    x=self.norm(x)
    logits=2*((ModelArgs.no_of_decoder_layers)**-0.5)*x
    #logits=self.linear_layer(logits)
    return logits, kv_cache

# KIMI-K2

In [ ]:
class Kimi_K2(nn.Module):
  def __init__(self, embed_dims=ModelArgs.embeddings_dims, device=ModelArgs.device, vocab_size=ModelArgs.vocab_size, block_size=ModelArgs.block_size, dropout=ModelArgs.dropout):
    super().__init__()
    self.embed_dims=embed_dims
    self.device=device
    self.vocab_size=vocab_size
    self.block_size=block_size
    self.dropout=dropout
    self.kv_cache=KVCache()
    self.decoder=Block(device=self.device, embeddings_dims=self.embed_dims, vocab_size=self.vocab_size, block_size=self.block_size, dropout=self.dropout)
    self.mpt_modules=nn.ModuleList([Decoder_Layer(device=self.device) for _ in range(ModelArgs.mtp_heads)])
    self.mpt_heads=nn.ModuleList([nn.Linear(self.embed_dims, self.embed_dims, bias=False, device=self.device) for _ in range(ModelArgs.mtp_heads)])
    self.embedding=nn.Embedding(self.vocab_size, self.embed_dims, dtype=torch.float32, device=self.device)
    self.norm1=RMSNorm(self.embed_dims, device=self.device)
    self.norm2=RMSNorm(self.embed_dims, device=self.device)
    self.linear_layer=nn.Linear(2*self.embed_dims, self.embed_dims, dtype=torch.float32, device=self.device)

    self.embedding.weight=self.decoder.embedding.weight

  def forward(self, x, inference=False, mask=None):
    if mask is not None:
      x=x*mask
    x=self.embedding(x)
    batch_size, seq_len, embed_dims=x.shape

    if inference:
      op, self.kv_cache=self.decoder(x, mask=mask, kv_cache=self.kv_cache)
      logits=self.decoder.linear_layer(op)
      return logits
    else:
    #   outputs=[]
    #   for i in range(seq_len-ModelArgs.mtp_heads):
    #     token_ops=[]
    #     for k in range(ModelArgs.mtp_heads):
    #       if k%ModelArgs.mtp_heads==0:
    #         h_z, _=self.decoder(x[:, i+k+1,:].unsqueeze(1), mask)

    #       else:
    #         h_z, _=self.mpt_modules[k](x[:, i+k+1,:].unsqueeze(1), mask)

    #       h_z=h_z.squeeze(1)
    #       embed=x[:,i+k+1, :]
    #       embed=self.norm1(embed)
    #       h_z=self.norm2(h_z)
    #       combined=torch.cat([embed, h_z], dim=-1)
    #       merged=self.linear_layer(combined)
    #       logits=self.decoder.linear_layer(merged)
    #       token_ops.append(logits)
    #     if token_ops:
    #       avg_output=torch.stack(token_ops, dim=1)
    #       outputs.append(avg_output)
    # final_op=torch.stack(outputs, dim=0)
    # final_op=final_op.permute(1,0,2,3)
      op, _=self.decoder(x, mask=mask)
      logits=self.decoder.linear_layer(op)
      return logits




In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary
# # Print summary to console

# Loading dataset

In [ ]:
from transformers import AutoTokenizer
from tokenizers import Tokenizer
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader, RandomSampler

In [ ]:
tinystories=True
fw=False
fw_train=None
fw_test=None


if (tinystories):
  fw_train=load_dataset("roneneldan/TinyStories", split="train", cache_dir="/tmp/hf_cache")
  fw_test=load_dataset("roneneldan/TinyStories", split="validation", cache_dir="/tmp/hf_cache")

if(fw):
    fw_train = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=False, cache_dir=None)
    fw_train = fw_train.train_test_split(test_size=0.01)
    print(fw_train)
    print(fw_train)

# Dataset creation

In [ ]:
class CreateDataset:
  def __init__(self, tokenizer, block_size):
    self.tokenizer=tokenizer
    self.block_size=block_size
    self.vocab_size=ModelArgs.vocab_size

  def collate_fn(self, batch):
    texts=[]
    for items in batch:
      tt=items["text"]
      texts.append(tt)
    input_encodings=self.tokenizer(texts, max_length=ModelArgs.block_size, padding="max_length", truncation=True, return_tensors="pt")
    input_encodings["labels"]=input_encodings["input_ids"].detach().clone()
    input_encodings["labels"][:,:-1]=input_encodings["input_ids"][:,1:]
    input_encodings["labels"][:, -1] = -100

    input_encodings["labels"][input_encodings["input_ids"] == self.tokenizer.pad_token_id] = -100


    return input_encodings

  def create_dataset(self, data_line,dataset_type, shuffle=False, use_sampler=False, drop_last=True, batch_size=32):
    if use_sampler:
      sampler=RandomSampler(data_line, replacement=False)
      shuffle=False
    else:
      sampler=None
    dataloader=DataLoader(
        data_line,
        batch_size=batch_size,
        shuffle=shuffle,
        sampler=sampler,
        drop_last=drop_last,
        collate_fn=self.collate_fn
    )

    return dataloader

def get_dataloader(tokenizer, ModelArgs, fw_train, fw_test, tinystories, split):
    """
    Create dataloader based on the split type

    Args:
        tokenizer: The tokenizer to use
        ModelArgs: Object containing block_size and batch_size
        fw_train: Training dataset
        fw_test: Test/validation dataset
        tinystories: Boolean flag for tinystories dataset
        split: "train" or "val"

    Returns:
        DataLoader object
    """
    dataset_creator = CreateDataset(tokenizer, ModelArgs.block_size)

    if tinystories:
        if split == "train":
            dataloader = dataset_creator.create_dataset(
                fw_train,
                dataset_type="train",
                shuffle=True,
                use_sampler=True,
                drop_last=True,
                batch_size=ModelArgs.batch_size
            )
        elif split == "val":
            dataloader = dataset_creator.create_dataset(
                fw_test,
                dataset_type="test",
                shuffle=False,
                use_sampler=True,
                drop_last=True,
                batch_size=ModelArgs.batch_size
            )
        else:
            raise ValueError(f"Unknown split: {split}. Expected 'train' or 'val'")
    else:
        raise ValueError("Currently only tinystories dataset is supported")

    return dataloader

# TopK

In [ ]:
def topk_sampling(model, tokenizer, prompt, device, max_length=50, top_k=50, temperature=1.0):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated_tokens = []

    if(len(input_ids[0]) < max_length):
        max_length -= len(input_ids[0]) # If the input is longer than max_length, set max_length to the length of the input
    else:
        max_length = len(input_ids[0]) - max_length
    for _ in range(max_length):
        with torch.no_grad(), torch.autocast(device_type=ModelArgs.device, dtype=torch.bfloat16):
            # Pass inference=True to use the inference path in the model
            outputs = model(input_ids, inference=True)
            logits = outputs[:, -1, :]
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)

            # Top-k filtering
            top_k_probs, top_k_indices = torch.topk(probs, top_k, dim=-1)

            # Sample from top-k
            next_token = torch.multinomial(top_k_probs, num_samples=1)

            xcol = torch.gather(top_k_indices, -1, next_token)
            input_ids = torch.cat([input_ids, xcol], dim=1) #1 because is it the dimension of the sequence

            if xcol.item() == tokenizer.eos_token_id:
                break

    model.kv_cache.cleanup()
    return tokenizer.decode(input_ids[0])


# Calculate GPU Memory

In [ ]:
class GPUMemoryCalculator:
    def __init__(self, model_args):
        self.model_args = model_args
        self.total_params = 378_935_808
        self.batch_size = model_args.batch_size
        self.block_size = model_args.block_size
        self.embeddings_dims = model_args.embeddings_dims
        self.vocab_size = model_args.vocab_size  # This seems wrong (100), should be ~50k+

    def calculate_model_memory(self):
        """Calculate memory for model parameters"""
        # Model parameters (FP16 training)
        model_params_fp16 = self.total_params * 2  # 2 bytes per param

        # Model parameters (FP32 master weights for optimizer)
        model_params_fp32 = self.total_params * 4  # 4 bytes per param

        return model_params_fp16, model_params_fp32

    def calculate_optimizer_memory(self):
        """Calculate memory for AdamW optimizer states"""
        # AdamW stores: momentum (m) + variance (v) + gradients
        # All in FP32
        momentum = self.total_params * 4      # m state
        variance = self.total_params * 4      # v state
        gradients = self.total_params * 4     # gradients

        return momentum + variance + gradients

    def calculate_activation_memory(self):
        """Calculate memory for forward/backward activations"""
        batch_size = self.batch_size
        seq_len = self.block_size
        hidden_dim = self.embeddings_dims
        num_layers = self.model_args.no_of_decoder_layers
        num_heads = self.model_args.no_of_heads

        # Input embeddings
        input_embeds = batch_size * seq_len * hidden_dim * 2  # FP16

        # Attention activations per layer (approximate)
        attn_per_layer = batch_size * num_heads * seq_len * seq_len * 2  # FP16
        attn_total = attn_per_layer * num_layers

        # Hidden states per layer
        hidden_per_layer = batch_size * seq_len * hidden_dim * 2  # FP16
        hidden_total = hidden_per_layer * num_layers * 2  # forward + backward

        # MoE expert activations (only top-k are active)
        experts_active = self.model_args.top_experts
        expert_hidden = hidden_dim * 4  # typical MLP expansion
        moe_activations = batch_size * seq_len * expert_hidden * experts_active * num_layers * 2

        return input_embeds + attn_total + hidden_total + moe_activations

    def calculate_total_memory(self):
        """Calculate total GPU memory requirement"""
        model_fp16, model_fp32 = self.calculate_model_memory()
        optimizer_memory = self.calculate_optimizer_memory()
        activation_memory = self.calculate_activation_memory()

        # Additional overhead (KV cache, temporary tensors, etc.)
        overhead = (model_fp16 + optimizer_memory + activation_memory) * 0.2

        total_bytes = model_fp16 + model_fp32 + optimizer_memory + activation_memory + overhead
        total_gb = total_bytes / (1024**3)

        return {
            'model_params_fp16_gb': model_fp16 / (1024**3),
            'model_params_fp32_gb': model_fp32 / (1024**3),
            'optimizer_states_gb': optimizer_memory / (1024**3),
            'activations_gb': activation_memory / (1024**3),
            'overhead_gb': overhead / (1024**3),
            'total_gb': total_gb,
            'recommended_gpu_gb': total_gb * 1.3  # 30% safety margin
        }

In [ ]:
calculator = GPUMemoryCalculator(ModelArgs())
memory_breakdown = calculator.calculate_total_memory()
print("🔥 GPU Memory Requirements for 379M Parameter MoE Model")
print("=" * 60)
print(f"📊 Model Parameters (FP16):     {memory_breakdown['model_params_fp16_gb']:.2f} GB")
print(f"📊 Model Parameters (FP32):     {memory_breakdown['model_params_fp32_gb']:.2f} GB")
print(f"🔧 Optimizer States:            {memory_breakdown['optimizer_states_gb']:.2f} GB")
print(f"⚡ Activations & Gradients:     {memory_breakdown['activations_gb']:.2f} GB")
print(f"💾 Overhead (20%):              {memory_breakdown['overhead_gb']:.2f} GB")
print("=" * 60)
print(f"💰 TOTAL MEMORY NEEDED:         {memory_breakdown['total_gb']:.2f} GB")
print(f"🎯 RECOMMENDED GPU MEMORY:      {memory_breakdown['recommended_gpu_gb']:.2f} GB")
print("=" * 60)

# GPU Recommendations
recommended_gb = memory_breakdown['recommended_gpu_gb']


🔥 GPU Memory Requirements for 379M Parameter MoE Model
📊 Model Parameters (FP16):     0.71 GB
📊 Model Parameters (FP32):     1.41 GB
🔧 Optimizer States:            4.23 GB
⚡ Activations & Gradients:     1.38 GB
💾 Overhead (20%):              1.26 GB
💰 TOTAL MEMORY NEEDED:         9.00 GB
🎯 RECOMMENDED GPU MEMORY:      11.70 GB


# Training

### Setup DDP

In [ ]:
import os
import tqdm

In [ ]:
def setup_ddp():
    """Initialize DDP setup"""
    # Initialize the process group
    dist.init_process_group(backend='nccl')

    # Get local rank from environment variable set by torchrun
    local_rank = int(os.environ['LOCAL_RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    rank = int(os.environ['RANK'])

    # Set device for this process
    torch.cuda.set_device(local_rank)
    device = torch.device(f'cuda:{local_rank}')

    return local_rank, world_size, rank, device

def cleanup_ddp():
    """Clean up DDP"""
    if dist.is_initialized():
        dist.destroy_process_group()


### Extras

In [ ]:
save_checkpoint_iter = 2000
total_iters = 10000 * ModelArgs.epochs
eval_iters = 400
eval_check = 400
warmup_iters = 400 * ModelArgs.epochs
min_lr = 0.1 * ModelArgs.max_lr
lr_decay_iters = 10000 * ModelArgs.epochs  # Total iterations for learning rate decay
total_batch_size = 524288
micro_batch_size = ModelArgs.batch_size
gradient_accumulation_steps = total_batch_size // (micro_batch_size * (ModelArgs.block_size * 1))


In [ ]:
torch.set_float32_matmul_precision('high')

#### Learning Rate

In [ ]:
def get_lr(it):
  if it<warmup_iters:
    return ModelArgs.max_lr*(it+1)/(warmup_iters+1)
  if it>lr_decay_iters:
    return min_lr
  decay_ratio=(it-warmup_iters)/(lr_decay_iters-warmup_iters)
  assert 0<=decay_ratio<=1
  coeff=0.5*(1.0+math.cos(math.pi*decay_ratio))
  return min_lr+coeff*(ModelArgs.max_lr-min_lr)

## Train Loop

### Muon

In [ ]:
def newton_schulz(G, steps:int):
  assert G.ndim>=2
  a,b,c=(3.4445, -4.7750, 2.0315)
  X=G.bfloat16()
  if G.size(-2)>G.size(-1):
    X=X.mT

  X=X/(X.norm(dim=(-2,-1), keepdim=True)+1e-7)
  for _ in range(steps):
    A=X @ X.mT
    B=b*A+c*A @ A
    X=a * X + B @ X

  if G.size(-2)>G.size(-1):
    X=X.mT
  return X

In [ ]:
def muon_update(grad, momentum, beta=0.95, ns_steps=5, nestrov=True):
  momentum.lerp_(grad,1-beta)
  update=grad.lerp_(momentum, beta) if nestrov else momentum
  if update.ndim==4:
    update=update.view(len(update), -1)
  update=newton_schulz(update, ns_steps)
  update*=max(1, grad.size(-2)/grad.size(-1))**0.5
  return update

In [ ]:
def adam_update(grad, buf1, buf2, step, betas, eps):
    buf1.lerp_(grad, 1 - betas[0])
    buf2.lerp_(grad.square(), 1 - betas[1])
    buf1c = buf1 / (1 - betas[0]**step)
    buf2c = buf2 / (1 - betas[1]**step)
    return buf1c / (buf2c.sqrt() + eps)

In [ ]:
import torch.distributed as dist
class Muon_Optimizer_withAuxAdam(torch.optim.Optimizer):
  def __init__(self, param_groups):
    for group in param_groups:
      if group["use_muon"]:
        group["params"]=sorted(group["params"], key=lambda x: x.size(), reverse=True)
        group["lr"]=group.get("lr", 0.02)
        group["momentum"]=group.get("momentum", 0.95)
        group["weight_decay"]=group.get("weight_decay", 0)
        assert set(group.keys())==set(["params", "lr", "momentum", "weight_decay", "use_muon"])
      else:
        group["lr"] = group.get("lr", 3e-4)
        group["betas"] = group.get("betas", (0.9, 0.95))
        group["eps"] = group.get("eps", 1e-10)
        group["weight_decay"] = group.get("weight_decay", 0)
        assert set(group.keys()) == set(["params", "lr", "betas", "eps", "weight_decay", "use_muon"])
    super().__init__(param_groups, dict())

  @torch.no_grad()
  def step(self, closure=None):
    loss=None
    if closure is not None:
      with torch.enable_grad():
        loss=closure()
    use_ddp=dist.is_initialized()
    for group in self.param_groups:
      if group["use_muon"]:
        params=group["params"]
        if use_ddp:
          params_pad=params+[torch.empty_like(params[-1])]*(dist.get_world_size()-len(params)%dist.get_world_size())
          for base_i in range(len(params))[::dist.get_world_size()]:
            if base_i+dist.get_rank()<len(params):
              p=params[base_i+dist.get_rank()]
              if p.grad is None:
                p.grad=torch.zeros_like(p)
              state=self.state[p]
              if len(state)==0:
                state["momentum_buffer"]=torch.zeros_like(p)
              update=muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"], ns_steps=5)
              p.mul_(1-group["lr"]*group["weight_decay"])
              p.add_(update.reshape(p.shape), alpha=-group["lr"])
            dist.all_gather(params_pad[base_i:base_i+dist.get_world_size()], params_pad[base_i+dist.get_rank()])

        else:
          for p in params:
            if p.grad is None:
              p.grad=torch.zeros_like(p)
            state=self.state[p]
            if len(state)==0:
              state["momentum_buffer"]=torch.zeros_like(p)
            update=muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"], ns_steps=5)
            p.mul_(1-group["lr"]*group["weight_decay"])
            p.add_(update.reshape(p.shape), alpha=-group["lr"])
      else:
        for p in group["params"]:
          if p.grad is None:
            p.grad=torch.zeros_like(p)
          state=self.state[p]
          if len(state)==0:
            state["step"]=0
            state["exp_avg"]=torch.zeros_like(p)
            state["exp_avg_sq"]=torch.zeros_like(p)
          state["step"]+=1
          update=adam_update(p.grad, state["exp_avg"], state["exp_avg_sq"], state["step"], group["betas"], group["eps"])
          p.mul_(1-group["lr"]*group["weight_decay"])
          p.add_(update, alpha=-group["lr"])
    return loss

### Training loop

In [ ]:
def train():
  use_ddp="RANK" in os.environ
  if use_ddp:
    local_rank, world_size, rank, device=setup_ddp()
  else:
    local_rank, world_size, rank, device=0, 1, 0, ModelArgs.device

  model=Kimi_K2(embed_dims=ModelArgs.embeddings_dims, vocab_size=ModelArgs.vocab_size, dropout=ModelArgs.dropout, block_size=ModelArgs.block_size)
  model.to(device)
  dataset_class=CreateDataset(tokenizer, ModelArgs.block_size)
  if use_ddp:
    model=torch.nn.parallel.DistributedDataParallel(model, device_ids=[local_rank], output_device=local_rank)
  base_model=model.module if use_ddp else model
  hidden_weights=[p for p in base_model.decoder.parameters() if p.ndim>=2]
  hidden_gains_biases=[p for p in base_model.decoder.parameters() if p.ndim<2]





  seen_params = set(id(p) for p in hidden_weights + hidden_gains_biases)

  non_hidden_params = []
  for p in base_model.embedding.parameters():
    if id(p) not in seen_params:
      non_hidden_params.append(p)
      seen_params.add(id(p))

  for p in base_model.linear_layer.parameters():
    if id(p) not in seen_params:
      non_hidden_params.append(p)
      seen_params.add(id(p))


  #non_hidden_params=[*base_model.linear_layer.parameters(), *base_model.embedding.parameters()]
  param_groups=[
      dict(params=hidden_weights, use_muon=True, lr=0.02,weight_decay=0.01),
      dict(params=hidden_gains_biases+non_hidden_params, use_muon=False, lr=ModelArgs.max_lr, betas=(ModelArgs.beta_1, ModelArgs.beta_2), weight_decay=ModelArgs.weight_decay_optim),
  ]
  optimizer=Muon_Optimizer_withAuxAdam(param_groups=param_groups)
  if rank==0:
    print("Model loaded")
  model=torch.compile(model)
  if rank==0:
    train_epoch_iterator=tqdm.tqdm(range(total_iters), desc="training")
  else:
    train_epoch_iterator=range(total_iters)
  val_dataloader=get_dataloader(tokenizer, ModelArgs, fw_train, fw_test, tinystories, "val")

  val_iter=iter(val_dataloader)

  @torch.inference_mode()
  def compute_loss():
    out = {}
    model.eval()
    count = 0
    for split in ['val']:
        if rank == 0:
            print(f"starting with {split} evaluation")
        losses = torch.zeros(eval_iters, device=device)
        for k in range(eval_iters):
            nonlocal val_iter
            try:
                batch = next(val_iter)
            except StopIteration:
                val_iter = iter(val_dataloader)
                batch = next(val_iter)

            idx = batch["input_ids"].to(device)
            targets = batch["labels"].to(device)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                eval_model = model.module if use_ddp else model
                if ModelArgs.use_liger:
                    ## Use Liger cross entropy loss
                    decoder_output = eval_model.decoder(eval_model.embedding(idx), mask=None)
                    decoder_out_flat = decoder_output.view(-1, ModelArgs.embeddings_dims)
                    target_flat = targets.view(-1)
                    if hasattr(eval_model, 'le_loss'):
                        loss = eval_model.le_loss(eval_model.linear_layer.weight, decoder_out_flat, target_flat)
                    else:
                        logits = eval_model.linear_layer(decoder_output)
                        B, T, C = logits.shape
                        logits_flat = logits.contiguous().view(B * T, C)
                        loss = F.cross_entropy(logits_flat, target_flat)
                else:
                    logits = eval_model(idx, mask=None, inference=False)
                    B, T, C = logits.shape
                    logits_flat = logits.contiguous().view(B * T, C)
                    targets_flat = targets.contiguous().view(-1)
                    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=-100, reduction='mean')

            eval_model.kv_cache.cleanup()
            losses[k] += loss.item()

        if use_ddp:
            dist.all_reduce(losses, op=dist.ReduceOp.AVG)
        else:
            out[split] = losses.mean().item()
    model.train()
    return out

  token_count=0
  model.train()
  if rank==0:
    print("Training starts")
    print("gradient steps: ", gradient_accumulation_steps)
  train_dataloader=get_dataloader(tokenizer, ModelArgs, fw_train, fw_test, tinystories, "train")
  train_iter=iter(train_dataloader)
  accumulated_loss=0
  if rank==0:
    print("Model compiled")
  for epoch in range(ModelArgs.epochs):
    if use_ddp and hasattr(train_dataloader.sampler, 'set_epoch'):
      train_dataloader.sampler.set_epoch(epoch)
    for step in train_epoch_iterator:
      if ((step%eval_iters==0 and step!=0) or step==total_iters-1) and rank==0:
        losses=compute_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

      ##Save checkpoints
      if step%save_checkpoint_iter==0 and step!=0 and rank==0:
        print(f"saving model for step: {step}")
        save_model=model.module if use_ddp else model
        torch.save(save_model.state_dict(), f"checkpoint_{step}.pt")
        print("checkpoint saved")

      accumulated_loss=0.0
      optimizer.zero_grad(set_to_none=True)

      for micro_step in range(gradient_accumulation_steps):
        try:
          batch=next(train_iter)
        except StopIteration:
          train_iter=iter(train_dataloader)
          batch=next(train_iter)

        idx=batch["input_ids"].to(device)
        targets=batch["labels"].to(device)
        token_count+=idx.numel()

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
          train_model=model.module if use_ddp else model
          if ModelArgs.use_liger:
            decoder_output=train_model.decoder(train_model.embedding(idx), mask=None)
            decoder_output_flat=decoder_output.view(-1, ModelArgs.embeddings_dims)
            target_flat=targets.view(-1)
            if hasattr(train_model, 'le_loss'):
                loss = train_model.le_loss(train_model.linear_layer.weight, decoder_output_flat, target_flat)
            else:
              logits=train_model.linear_layer(decoder_output)
              B,T,C=logits.shape
              logits_flat=logits.contiguous().view(-1,C)
              loss=F.cross_entropy(logits_flat, target_flat,
                                   ignore_index=-100,#tokenizer.pad_token_id,
                                   reduction='mean')

          else:
            logits=train_model(idx, mask=None)
            B,T,C=logits.shape
            logit_flat=logits.contiguous().view(-1, C)
            targets_flat=targets.contiguous().view(-1)
            print(f"Logits shape: {logit_flat.shape}")  # Should be [B*T, vocab_size]
            print(f"Targets shape: {targets_flat.shape}")  # Should be [B*T]
            print(f"Targets min: {targets_flat.min()}, max: {targets_flat.max()}")
            print(f"Vocab size: {ModelArgs.vocab_size}")
            print(f"Number of classes in logits: {logit_flat.shape[-1]}")

            loss=F.cross_entropy(
                logit_flat, targets_flat,
                ignore_index=-100,#tokenizer.pad_token_id,
                reduction='mean'
            )
        loss=loss/gradient_accumulation_steps
        loss.backward()
        accumulated_loss+=loss
        if micro_step%10==0 and rank==0:
          print(f"Micro Batch: {micro_step}/{gradient_accumulation_steps}")
          print(f"Step: {step}/{total_iters}")
          print(f"Total tokens processed: {token_count}")

        #synchronise the accumulated loss across the gpus
        if use_ddp:
          loss_tensor=torch.tensor(accumulated_loss, device=device)
          dist.all_reduce(loss_tensor, op=dist.ReduceOp.AVG)
          accumulated_loss=loss_tensor.item()


        #Update learning rate
        lr=get_lr(step)
        for params in optimizer.param_groups:
          params['lr']=lr

        #Gradient clipping

        grad_norm_value=0
        if ModelArgs.clip!=0:
          torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=ModelArgs.clip)


        optimizer.step()

    if use_ddp:
      cleanup_ddp()


In [ ]:
world_size = torch.cuda.device_count()
print(f"CUDA devices available: {world_size}")


CUDA devices available: 2


In [ ]:
train()

Model loaded


training:   0%|          | 0/10000 [00:00<?, ?it/s]

Training starts
gradient steps:  64
Model compiled


W1003 21:05:07.433000 282 torch/_inductor/utils.py:1137] [2/0_1] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:1948: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:1948: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:1948: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:1948: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:1948: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.11/dist

Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Micro Batch: 0/64
Step: 0/10000
Total tokens processed: 8192
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 320

training:   0%|          | 1/10000 [04:14<706:37:09, 254.41s/it]

Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Micro Batch: 0/64
Step: 1/10000
Total tokens processed: 532480
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 3

training:   0%|          | 2/10000 [08:21<694:41:43, 250.14s/it]

Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Micro Batch: 0/64
Step: 2/10000
Total tokens processed: 1056768
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 32001
Logits shape: torch.Size([8192, 32001])
Targets shape: torch.Size([8192])
Targets min: -100, max: 32000
Vocab size: 32001
Number of classes in logits: 

In [ ]:
print(f"Pad token: {tokenizer.pad_token}")
print(f"Pad token ID: {tokenizer.pad_token_id}")
